# Solar Power Forecasting using Recurrent Neural Network (GRU)

- **Student Name:** DUDEKULA THAREEQ
- **Registration No:** 2023008156
- **Domain:** Renewable Energy
- **Unit / Outcome:** Unit 3 | CO3
- **Dataset:** UK Deterministic Solar Forecasting Dataset (Open Climate Fix / Met Office)
- **Assigned Architecture:** Gated Recurrent Unit (GRU)
- **Assigned Optimization:** Mini-Batch Gradient Descent
- **Level:** Advanced

---
### Assignment Objectives:
1. Download and inspect multivariate weather/NWP variables.
2. Preprocess and chronologically split without data leakage.
3. Implement assigned GRU neural architecture.
4. Train with Mini-Batch Gradient Descent (batch size = 32).
5. Construct chronological windows and compare GRU sequence lengths ($L = 6, 12, 24$ hours).
6. Compare against simple baselines (Persistence model and Linear Regression).
7. Report metrics (MAE, RMSE, $R^2$), plot forecast curves, and document observations.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow Version:", tf.__version__)

## 1. Dataset Loading and Inspection
We load the UK Deterministic Solar Forecasting Dataset containing NWP weather features and regional aggregate Solar PV generation.

In [ ]:
from data_loader import load_solar_nwp_dataset, prepare_chronological_splits, create_sliding_windows

df = load_solar_nwp_dataset("met_office_uk_deterministic_solar_data.csv")
display(df.head())
display(df.describe())

## 2. Preprocessing & Chronological Splitting (No Data Leakage)
We strictly avoid shuffling to prevent look-ahead bias:
- **Train Set:** First 70%
- **Validation Set:** Next 15%
- **Test Set:** Final 15%
- Feature and target scalers are fitted **ONLY** on the training split.

In [ ]:
feature_cols = [
    "surface_solar_radiation_downwards_wm2",
    "temperature_2m_degC",
    "total_cloud_cover_percent",
    "relative_humidity_percent",
    "wind_speed_10m_mps",
    "solar_zenith_angle_deg",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos"
]
target_col = "solar_pv_generation_mw"

splits = prepare_chronological_splits(df, feature_cols, target_col, train_ratio=0.70, val_ratio=0.15)
scaler_x, scaler_y = splits["scaler_x"], splits["scaler_y"]
X_train, y_train = splits["X_train_scaled"], splits["y_train_scaled"]
X_val, y_val = splits["X_val_scaled"], splits["y_val_scaled"]
X_test, y_test = splits["X_test_scaled"], splits["y_test_scaled"]

## 3. GRU Architecture and Mini-Batch Gradient Descent Training
We evaluate three historical sequence lookback lengths:
- $L = 6$ hours (immediate weather transients)
- $L = 12$ hours (half-day solar arc)
- $L = 24$ hours (full diurnal solar cycle)

In [ ]:
from gru_model import build_gru_model

seq_lengths = [6, 12, 24]
batch_size = 32
epochs = 25

results = []
histories = {}
predictions = {}
actuals = {}

for L in seq_lengths:
    print(f"\n--- Training GRU with Sequence Length L={L} hours ---")
    X_tr_w, y_tr_w = create_sliding_windows(X_train, y_train, seq_length=L, horizon=1)
    X_va_w, y_va_w = create_sliding_windows(X_val, y_val, seq_length=L, horizon=1)
    X_te_w, y_te_w = create_sliding_windows(X_test, y_test, seq_length=L, horizon=1)
    
    model = build_gru_model(seq_length=L, n_features=len(feature_cols))
    early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
    
    h = model.fit(X_tr_w, y_tr_w, validation_data=(X_va_w, y_va_w),
                  epochs=epochs, batch_size=batch_size, callbacks=[early_stop], verbose=1)
    
    pred_scaled = model.predict(X_te_w)
    pred_mw = np.maximum(0.0, scaler_y.inverse_transform(pred_scaled).ravel())
    true_mw = scaler_y.inverse_transform(y_te_w).ravel()
    
    mae = mean_absolute_error(true_mw, pred_mw)
    rmse = np.sqrt(mean_squared_error(true_mw, pred_mw))
    r2 = r2_score(true_mw, pred_mw)
    
    tag = f"GRU (L={L}h)"
    results.append({"Model": tag, "MAE (MW)": round(mae, 3), "RMSE (MW)": round(rmse, 3), "R2": round(r2, 4)})
    histories[L] = h.history
    predictions[tag] = pred_mw
    actuals[L] = true_mw

## 4. Benchmark against Simple Baselines
- **Persistence Baseline:** $y_{pred}[t+1] = y[t]$
- **Linear Regression Baseline:** Multivariate ordinary least squares regression.

In [ ]:
from gru_model import LinearRegressionBaseline

L_ref = 24
X_tr_ref, y_tr_ref = create_sliding_windows(X_train, y_train, seq_length=L_ref, horizon=1)
X_te_ref, y_te_ref = create_sliding_windows(X_test, y_test, seq_length=L_ref, horizon=1)
true_mw_ref = actuals[L_ref]

# Persistence Baseline
raw_test_targets = splits["test_df"][target_col].values
pers_pred = raw_test_targets[L_ref-1 : -1][:len(true_mw_ref)]
results.append({
    "Model": "Persistence Baseline",
    "MAE (MW)": round(mean_absolute_error(true_mw_ref, pers_pred), 3),
    "RMSE (MW)": round(np.sqrt(mean_squared_error(true_mw_ref, pers_pred)), 3),
    "R2": round(r2_score(true_mw_ref, pers_pred), 4)
})
predictions["Persistence Baseline"] = pers_pred

# Linear Regression Baseline
lr_model = LinearRegressionBaseline()
lr_model.fit(X_tr_ref, y_tr_ref)
lr_pred = np.maximum(0.0, scaler_y.inverse_transform(lr_model.predict(X_te_ref)).ravel())
results.append({
    "Model": "Linear Regression Baseline",
    "MAE (MW)": round(mean_absolute_error(true_mw_ref, lr_pred), 3),
    "RMSE (MW)": round(np.sqrt(mean_squared_error(true_mw_ref, lr_pred)), 3),
    "R2": round(r2_score(true_mw_ref, lr_pred), 4)
})
predictions["Linear Regression"] = lr_pred

# Summary Table
benchmark_df = pd.DataFrame(results)
display(benchmark_df)

## 5. Forecast Curves and Loss Visualization

In [ ]:
# Forecast Curves (120 Hours / 5 Days)
plt.figure(figsize=(14, 6), dpi=150)
plt.plot(true_mw_ref[:120], label="Actual Solar PV Generation", color="black", linewidth=2.5)
plt.plot(predictions["GRU (L=24h)"][:120], label="GRU (L=24h)", color="green", linewidth=2.0)
plt.plot(predictions["GRU (L=6h)"][:120], label="GRU (L=6h)", color="blue", linestyle=":", linewidth=1.5)
plt.plot(predictions["Persistence Baseline"][:120], label="Persistence Baseline", color="red", linestyle="--", linewidth=1.2)
plt.title("Solar PV Power Forecast Curves (120 Hours Horizon)", fontsize=14, fontweight='bold')
plt.xlabel("Hours")
plt.ylabel("Solar Generation (MW)")
plt.legend()
plt.grid(True, linestyle=":")
plt.show()